In [1]:
import os
import sys
import time
import threading
import queue
import struct
from pynq import allocate
import numpy as np
import cv2
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600

WORK_DIR = '/home/xilinx/jupyter_notebooks/duxu/pynq_vqvae'
PL_DIR = os.path.join(WORK_DIR, 'zcu111_index_dequant')
PRE_DIR = '/home/xilinx/jupyter_notebooks/duxu/pynq_vqvae/imgs_preprocessed'
IMG_DIR = '/home/xilinx/jupyter_notebooks/duxu/pynq_vqvae/imgs'
CODEBOOK_PATH = os.path.join(WORK_DIR, 'codebook.npy')
ENC_XMODEL = os.path.join(WORK_DIR, 'xmodel/encoder_768x512.xmodel')
DEC_XMODEL = os.path.join(WORK_DIR, 'xmodel/decoder_zcu111_upsample.xmodel')
RES_DIR = './results_768x512'
IDX_DIR = os.path.join(RES_DIR, 'idx_bins')

sys.path.append('/usr/lib/python3/site-packages')
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/soft/DPU-PYNQ')

from pynq_dpu import DpuOverlay
import vart
import xir

if not os.path.exists(RES_DIR):
    os.makedirs(RES_DIR)
if not os.path.exists(IDX_DIR):
    os.makedirs(IDX_DIR)

In [2]:
# === 量化参数（与 run_in_one.py / parallel_600dpi.ipynb 一致）===
enc_out_scale = 0.015625   # encoder 输出 fix=6
dec_in_scale  = 0.03125    # decoder 输入 fix=5
dec_out_scale = 0.007812   # decoder 输出 fix=7
enc_in_scale  = 0.015625   # encoder 输入量化 scale（与预处理一致）
dec_scale_inv = 1.0 / dec_in_scale

num_vectors = 128 * 192
dim = 64
num_code = 512
num_bufs = 3
num_pingpong = 2

TARGET_W, TARGET_H = 768, 512

def set_u64(mmio, lo_off, hi_off, addr):
    mmio.write(lo_off, addr & 0xFFFFFFFF)
    mmio.write(hi_off, (addr >> 32) & 0xFFFFFFFF)

def write_float(mmio, off, value):
    mmio.write(off, struct.unpack('<I', struct.pack('<f', np.float32(value)))[0])

def start_and_wait_old_style(mmio, timeout_s=2.0):
    mmio.write(0x00, 0x11)
    t0 = time.time()
    while (mmio.read(0x00) & 0x02) == 0:
        if time.time() - t0 > timeout_s:
            raise RuntimeError("IP timeout waiting for AP_DONE")
        time.sleep(0.0001)

def get_dpu_subgraph(path):
    graph = xir.Graph.deserialize(path)
    return graph, graph.get_root_subgraph().toposort_child_subgraph()[1]

In [3]:
# === DPU overlay 和 runners ===
overlay = DpuOverlay(os.path.join(PL_DIR, 'dpu.bit'))
vq_accel = overlay.vq_accel_1
vq_dequant = overlay.vq_dequant_1

_, enc_subgraph = get_dpu_subgraph(ENC_XMODEL)
enc_runner = vart.Runner.create_runner(enc_subgraph, "run")

_, dec_subgraph = get_dpu_subgraph(DEC_XMODEL)
dec_runner = vart.Runner.create_runner(dec_subgraph, "run")

# Encoder output buffers
enc_feat_bufs = [
    allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)
    for _ in range(num_bufs)
]

# Phase1 staging
vq1_stage_in_buf = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)
vq1_stage_idx_buf = allocate(shape=(num_vectors,), dtype=np.uint16, cacheable=1)

# Phase2 ping-pong
vq2_stage_idx_bufs = [
    allocate(shape=(num_vectors,), dtype=np.uint16, cacheable=1)
    for _ in range(num_pingpong)
]
vq2_stage_zq_bufs = [
    allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=1)
    for _ in range(num_pingpong)
]

# Decoder output ping-pong
dec_out_bufs = [
    np.empty((1, TARGET_H, TARGET_W, 3), dtype=np.int8, order='C')
    for _ in range(num_pingpong)
]

# Codebook
codebook = np.load(CODEBOOK_PATH).astype(np.float32)
assert codebook.shape == (num_code, dim), "codebook shape mismatch"

vq_codebook_buf = allocate(shape=(num_code, dim), dtype=np.float32, cacheable=1)
vq_codebook_buf[:] = codebook
vq_codebook_buf.sync_to_device()

set_u64(vq_accel.mmio, 0x1C, 0x20, vq_codebook_buf.device_address)
write_float(vq_accel.mmio, 0x34, enc_out_scale)
write_float(vq_accel.mmio, 0x3C, dec_scale_inv)

set_u64(vq_dequant.mmio, 0x1C, 0x20, vq_codebook_buf.device_address)
write_float(vq_dequant.mmio, 0x34, dec_scale_inv)

print("DPU overlay 和 runners 初始化完成")

DPU overlay 和 runners 初始化完成


In [4]:
# === 加载输入数据 ===
data_files = sorted([f for f in os.listdir(PRE_DIR) if f.endswith('.npy')])
num_imgs = len(data_files)
print(f"共 {num_imgs} 张预处理图片")

# LUT 后处理
post_lut = np.zeros(256, dtype=np.uint8)
for i in range(256):
    val_int8 = np.int8(i)
    val_fp32 = float(val_int8) * dec_out_scale
    val_norm = max(0.0, min(1.0, val_fp32 * 0.5 + 0.5))
    post_lut[i] = int(val_norm * 255.0)

# 队列
read_queue = queue.Queue(maxsize=3)
free_queue = queue.Queue(maxsize=num_bufs)
enc_res_queue = queue.Queue(maxsize=num_bufs)

for i in range(num_bufs):
    free_queue.put(i)

all_idx_results = [None] * num_imgs
all_recon_imgs = [None] * num_imgs

共 10 张预处理图片


In [5]:
# === Phase 1 workers ===
def read_worker():
    for img_id, f in enumerate(data_files):
        data = np.load(os.path.join(PRE_DIR, f))
        read_queue.put((img_id, data))
    read_queue.put(None)

def enc_worker():
    while True:
        item = read_queue.get()
        if item is None:
            enc_res_queue.put(None)
            break
        img_id, input_data = item
        in_buf = [np.ascontiguousarray(input_data[np.newaxis])]
        buf_idx = free_queue.get()
        target_feat_buf = enc_feat_bufs[buf_idx]
        out_buf = [np.ndarray((1, 128, 192, 64), dtype=np.int8, buffer=target_feat_buf.data)]
        job_id = enc_runner.execute_async(in_buf, out_buf)
        enc_runner.wait(job_id)
        enc_res_queue.put((img_id, buf_idx))

def phase1_pipeline():
    print(f"[Phase 1] Read -> Encoder -> vq_accel: {num_imgs} images")
    t_read = threading.Thread(target=read_worker)
    t_enc  = threading.Thread(target=enc_worker)
    start_time = time.time()
    t_read.start()
    t_enc.start()
    while True:
        item = enc_res_queue.get()
        if item is None:
            break
        img_id, buf_idx = item
        curr_feat_buf = enc_feat_bufs[buf_idx]
        curr_feat_buf.sync_from_device()
        vq1_stage_in_buf[:] = curr_feat_buf
        vq1_stage_in_buf.sync_to_device()
        vq1_stage_idx_buf[:] = 0
        vq1_stage_idx_buf.sync_to_device()
        set_u64(vq_accel.mmio, 0x10, 0x14, vq1_stage_in_buf.device_address)
        set_u64(vq_accel.mmio, 0x28, 0x2C, vq1_stage_idx_buf.device_address)
        start_and_wait_old_style(vq_accel.mmio)
        vq1_stage_idx_buf.sync_from_device()
        idx_snapshot = np.array(vq1_stage_idx_buf, copy=True)
        all_idx_results[img_id] = idx_snapshot
        idx_path = os.path.join(IDX_DIR, f'idx_{img_id:04d}.bin')
        with open(idx_path, 'wb') as f:
            f.write(idx_snapshot.tobytes())
        free_queue.put(buf_idx)
    t_read.join()
    t_enc.join()
    phase1_time = time.time() - start_time
    print(f"[Phase 1] Finished! {phase1_time*1000:.0f} ms | FPS: {num_imgs/phase1_time:.1f}")

phase1_pipeline()

[Phase 1] Read -> Encoder -> vq_accel: 10 images
[Phase 1] Finished! 288 ms | FPS: 34.8


In [6]:
# === Phase 2 workers ===
deq_in_queue  = queue.Queue(maxsize=3)
dec_in_queue  = queue.Queue(maxsize=3)
dec_out_queue = queue.Queue(maxsize=3)

free_deq_slots = queue.Queue(maxsize=num_pingpong)
free_dec_slots = queue.Queue(maxsize=num_pingpong)
for i in range(num_pingpong):
    free_deq_slots.put(i)
    free_dec_slots.put(i)

def feed_idx_worker():
    for img_id, idx_res in enumerate(all_idx_results):
        deq_in_queue.put((img_id, idx_res))
    deq_in_queue.put(None)

def vq_dequant_worker():
    while True:
        item = deq_in_queue.get()
        if item is None:
            dec_in_queue.put(None)
            break
        img_id, idx_res = item
        if idx_res is None:
            dec_in_queue.put((img_id, None, None))
            continue
        slot = free_deq_slots.get()
        idx_buf = vq2_stage_idx_bufs[slot]
        zq_buf  = vq2_stage_zq_bufs[slot]
        idx_buf[:] = idx_res
        idx_buf.sync_to_device()
        zq_buf[:] = 0
        zq_buf.sync_to_device()
        set_u64(vq_dequant.mmio, 0x10, 0x14, idx_buf.device_address)
        set_u64(vq_dequant.mmio, 0x28, 0x2C, zq_buf.device_address)
        start_and_wait_old_style(vq_dequant.mmio)
        zq_buf.sync_from_device()
        dec_in_queue.put((img_id, slot, zq_buf))

def dec_worker():
    while True:
        item = dec_in_queue.get()
        if item is None:
            dec_out_queue.put(None)
            break
        img_id, deq_slot, zq_buf = item
        if zq_buf is None:
            dec_out_queue.put((img_id, None))
            continue
        dec_slot = free_dec_slots.get()
        z_q_int8 = zq_buf.reshape(1, 128, 192, 64)
        dec_in_buf  = [z_q_int8]
        dec_out_buf = [dec_out_bufs[dec_slot]]
        job_id = dec_runner.execute_async(dec_in_buf, dec_out_buf)
        dec_runner.wait(job_id)
        free_deq_slots.put(deq_slot)
        dec_out_queue.put((img_id, dec_slot))

def lut_worker():
    while True:
        item = dec_out_queue.get()
        if item is None:
            break
        img_id, dec_slot = item
        if dec_slot is None:
            continue
        recon_img = post_lut[dec_out_bufs[dec_slot][0].view(np.uint8)]
        all_recon_imgs[img_id] = recon_img
        free_dec_slots.put(dec_slot)

def phase2_pipeline():
    print(f"[Phase 2] vq_dequant -> Decoder -> LUT: {num_imgs} images")
    t_feed = threading.Thread(target=feed_idx_worker)
    t_vq   = threading.Thread(target=vq_dequant_worker)
    t_dec  = threading.Thread(target=dec_worker)
    t_lut  = threading.Thread(target=lut_worker)
    start_time = time.time()
    t_feed.start()
    t_vq.start()
    t_dec.start()
    t_lut.start()
    t_feed.join()
    t_vq.join()
    t_dec.join()
    t_lut.join()
    phase2_time = time.time() - start_time
    print(f"[Phase 2] Finished! {phase2_time*1000:.0f} ms | FPS: {num_imgs/phase2_time:.1f}")

phase2_pipeline()

del enc_runner
del dec_runner
print("DPU runners released.")

[Phase 2] vq_dequant -> Decoder -> LUT: 10 images
[Phase 2] Finished! 425 ms | FPS: 23.6
DPU runners released.


In [7]:
# === 保存重建图像 ===
print("Writing final images with DPI=600...")
saved = 0
for i, img in enumerate(all_recon_imgs):
    if img is None:
        continue
    h, w = img.shape[:2]
    save_path = f'{RES_DIR}/recon_{i}.png'
    fig = plt.figure(figsize=(w/600, h/600), dpi=600)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(img)
    ax.axis('off')
    plt.savefig(save_path, dpi=600, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    saved += 1
print(f"Saved {saved} images to {RES_DIR}/")

Writing final images with DPI=600...
Saved 10 images to ./results_768x512/


In [8]:
mapping_path = os.path.join(RES_DIR, "recon_mapping.txt")

with open(mapping_path, "w") as mf:
    print("Writing final images with DPI=600...")
    saved = 0

    for i, img in enumerate(all_recon_imgs):
        if img is None:
            continue

        src_npy = data_files[i]
        src_base = src_npy

        if src_base.endswith(".npy"):
            src_base = src_base[:-4]

        safe_name = src_base.replace("/", "_").replace(" ", "_")
        save_name = f"recon_{i:04d}_{safe_name}.png"
        save_path = os.path.join(RES_DIR, save_name)

        h, w = img.shape[:2]
        fig = plt.figure(figsize=(w / 600, h / 600), dpi=600)
        ax = fig.add_axes([0, 0, 1, 1])
        ax.imshow(img)
        ax.axis("off")
        plt.savefig(save_path, dpi=600, bbox_inches="tight", pad_inches=0)
        plt.close(fig)

        mf.write(f"{i}\t{save_name}\t{src_npy}\n")
        saved += 1

print(f"Saved {saved} images to {RES_DIR}/")
print(f"Saved mapping to: {mapping_path}")

Writing final images with DPI=600...
Saved 10 images to ./results_768x512/
Saved mapping to: ./results_768x512/recon_mapping.txt


In [9]:
# ============================================================
# SSIM / PSNR 评估
# ============================================================
from PIL import Image as PILImage

def ssim_numpy(img1, img2, win_size=7):
    """通道级 SSIM，img1/img2 均为 float32 [0,1] 范围"""
    C1 = (0.01) ** 2
    C2 = (0.03) ** 2
    ssim_channels = []
    for c in range(img1.shape[2]):
        a = img1[:, :, c].astype(np.float64)
        b = img2[:, :, c].astype(np.float64)
        mu_a  = a.mean()
        mu_b  = b.mean()
        sig_a = a.var()
        sig_b = b.var()
        sig_ab = np.mean((a - mu_a) * (b - mu_b))
        num = (2*mu_a*mu_b + C1) * (2*sig_ab + C2)
        den = (mu_a**2 + mu_b**2 + C1) * (sig_a + sig_b + C2)
        ssim_channels.append(num / den)
    return float(np.mean(ssim_channels))

def psnr_numpy(img1, img2):
    mse = np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')
    return 20.0 * np.log10(1.0 / np.sqrt(mse))

print("\n" + "="*55)
print("  FPGA Decoder 输出 — SSIM / PSNR 评估")
print("="*55)

img_files = sorted([f for f in os.listdir(IMG_DIR)
                    if f.lower().endswith(('.png','.jpg','.jpeg'))])

ssim_list = []
psnr_list = []

for i in range(num_imgs):
    recon = all_recon_imgs[i]
    if recon is None:
        continue

    # 加载 ground truth（与预处理同样的 resize）
    if i < len(img_files):
        gt_pil = PILImage.open(os.path.join(IMG_DIR, img_files[i])).convert('RGB')
        gt_arr = np.asarray(gt_pil.resize((TARGET_W, TARGET_H), PILImage.BILINEAR),
                            dtype=np.float32) / 255.0
    else:
        print(f"  警告: 图 {i} 无对应原图，跳过")
        continue

    # 重建图已在 all_recon_imgs 中为 uint8 [0,255]
    recon_f32 = recon.astype(np.float32) / 255.0

    ssim_val = ssim_numpy(gt_arr, recon_f32)
    psnr_val = psnr_numpy(gt_arr, recon_f32)
    ssim_list.append(ssim_val)
    psnr_list.append(psnr_val)
    print(f"  img {i:3d}: SSIM={ssim_val:.4f}  PSNR={psnr_val:.2f} dB")

print("-"*55)
print(f"  Average: SSIM={np.mean(ssim_list):.4f}  "
      f"PSNR={np.mean(psnr_list):.2f} dB  (n={len(ssim_list)})")
print("="*55)


  FPGA Decoder 输出 — SSIM / PSNR 评估
  img   0: SSIM=0.9331  PSNR=25.20 dB
  img   1: SSIM=0.8858  PSNR=19.60 dB
  img   2: SSIM=0.9632  PSNR=26.67 dB
  img   3: SSIM=0.8849  PSNR=22.05 dB
  img   4: SSIM=0.9213  PSNR=26.53 dB
  img   5: SSIM=0.9048  PSNR=21.78 dB
  img   6: SSIM=0.8741  PSNR=23.74 dB
  img   7: SSIM=0.9264  PSNR=21.66 dB
  img   8: SSIM=0.9200  PSNR=22.14 dB
  img   9: SSIM=0.9487  PSNR=24.10 dB
-------------------------------------------------------
  Average: SSIM=0.9162  PSNR=23.35 dB  (n=10)


## 结果汇总

| 指标 | 值 |
|------|-----|
| Decoder 架构 | Upsample(nearest) + Conv2d(k=3,s=1) |
| 量化精度 | INT8 (DPU) |
| 对比基准 | 原图 resize 到 768×512 |